# Descripción del archivo `01_eda_inicial.ipynb`

Este notebook tiene como objetivo realizar un Análisis Exploratorio de Datos (EDA) inicial sobre el dataset seleccionado, con el fin de comprender su estructura, contenido y calidad.

En este archivo se explorarán aspectos clave como la cantidad de registros, tipos de variables, valores faltantes, duplicados, distribución de datos y características principales de las columnas más relevantes.

El propósito principal no es limpiar los datos, sino identificar posibles problemas y oportunidades de mejora, para así definir las acciones de limpieza y transformación que se implementarán en etapas posteriores del proyecto.

Este análisis permitirá establecer una base sólida para el procesamiento de datos y el desarrollo de análisis más avanzados.


## Carga del dataset

En esta sección se carga el dataset utilizando una función personalizada que permite obtener los datos desde la ruta local o descargarlos automáticamente si no existen. 

Los datos son almacenados en un DataFrame de pandas para facilitar su exploración y análisis inicial.

In [9]:
from pathlib import Path
import sys

# Encontrar el raíz del proyecto buscando pyproject.toml o requirements.txt
current = Path.cwd()
while current != current.parent:
    if (current / "pyproject.toml").exists() or (current / "requirements.txt").exists():
        project_root = current
        break
    current = current.parent
else:
    project_root = Path.cwd()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.load_data import load_medicine_data
df = load_medicine_data(download_if_missing=True)
df.head()

,Medicine Name,Composition,Uses,Side_effects,Image URL,Manufacturer,Excellent Review %,Average Review %,Poor Review %
0,Avastin 400mg Injection,Bevacizumab (400mg),Cancer of colon and rectum Non-small cell lun...,Rectal bleeding Taste change Headache Noseblee...,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Roche Products India Pvt Ltd,22,56,22
1,Augmentin 625 Duo Tablet,Amoxycillin (500mg) + Clavulanic Acid (125mg),Treatment of Bacterial infections,Vomiting Nausea Diarrhea Mucocutaneous candidi...,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Glaxo SmithKline Pharmaceuticals Ltd,47,35,18
2,Azithral 500 Tablet,Azithromycin (500mg),Treatment of Bacterial infections,Nausea Abdominal pain Diarrhea,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Alembic Pharmaceuticals Ltd,39,40,21
3,Ascoril LS Syrup,Ambroxol (30mg/5ml) + Levosalbutamol (1mg/5ml)...,Treatment of Cough with mucus,Nausea Vomiting Diarrhea Upset stomach Stomach...,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Glenmark Pharmaceuticals Ltd,24,41,35
4,Aciloc 150 Tablet,Ranitidine (150mg),Treatment of Gastroesophageal reflux disease (...,Headache Diarrhea Gastrointestinal disturbance,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Cadila Pharmaceuticals Ltd,34,37,29


## Resumen del dataset

Se presenta un resumen estructurado del dataset, incluyendo nombres de columnas, tipos de datos y cantidad de valores nulos y no nulos.

Esto permite tener una visión clara y organizada de la calidad y estructura de los datos antes de continuar con el análisis.

In [10]:
import pandas as pd

# Crear resumen del dataset
resumen = pd.DataFrame({
    "Columna": df.columns,
    "Tipo de dato": df.dtypes.values,
})

# Mostrar dimensiones
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

# Mostrar tabla ordenada
resumen

Filas: 11825
Columnas: 9


,Columna,Tipo de dato
0,Medicine Name,str
1,Composition,str
2,Uses,str
3,Side_effects,str
4,Image URL,str
5,Manufacturer,str
6,Excellent Review %,int64
7,Average Review %,int64
8,Poor Review %,int64


## Revisión de valores faltantes

En esta sección se analizan los valores nulos y vacíos del dataset para evaluar su calidad.

Se identifican posibles problemas de datos incompletos que podrían afectar el análisis y que deberán ser considerados en la etapa de limpieza.

In [11]:
# Conteo de valores nulos por columna
nulos = df.isnull().sum()

# Porcentaje de nulos
porcentaje_nulos = (df.isnull().sum() / len(df)) * 100

# Crear resumen
nulos_df = pd.DataFrame({
    "Columna": df.columns,
    "Nulos": nulos.values,
    "Porcentaje (%)": porcentaje_nulos.values
})

nulos_df

,Columna,Nulos,Porcentaje (%)
0,Medicine Name,0,0.0
1,Composition,0,0.0
2,Uses,0,0.0
3,Side_effects,0,0.0
4,Image URL,0,0.0
5,Manufacturer,0,0.0
6,Excellent Review %,0,0.0
7,Average Review %,0,0.0
8,Poor Review %,0,0.0


In [12]:
# Detectar strings vacíos
vacios = (df == "").sum()

vacios_df = pd.DataFrame({
    "Columna": df.columns,
    "Valores vacíos": vacios.values
})

vacios_df

,Columna,Valores vacíos
0,Medicine Name,0
1,Composition,0
2,Uses,0
3,Side_effects,0
4,Image URL,0
5,Manufacturer,0
6,Excellent Review %,0
7,Average Review %,0
8,Poor Review %,0


## Revisión de duplicados

En esta sección se analizan los duplicados completos del dataset, es decir, aquellos registros que presentan exactamente los mismos valores en todas sus columnas.

Se consideran únicamente duplicados completos, ya que representan una repetición exacta de la información y no aportan valor al análisis, por lo que pueden ser eliminados sin afectar la integridad de los datos.

No se eliminan registros duplicados en columnas específicas (como el nombre del medicamento), debido a que estos pueden corresponder a diferentes composiciones, fabricantes o valoraciones, lo cual es relevante para el análisis y no debe ser considerado un error.

In [13]:
# Cantidad de duplicados completos
duplicados_totales = df.duplicated().sum()

print(f"Cantidad de duplicados completos: {duplicados_totales}")

Cantidad de duplicados completos: 84


## Análisis de composiciones de medicamentos

En esta sección se analizan las composiciones químicas de los medicamentos con el objetivo de identificar posibles equivalencias entre ellos.

Dado que la columna de composición contiene múltiples componentes en una misma celda y con distintos formatos, se realiza una normalización del texto, separando los componentes, eliminando espacios innecesarios y ordenándolos de forma consistente.

Posteriormente, se agrupan los registros por composición normalizada para identificar medicamentos que comparten los mismos principios activos, permitiendo detectar posibles sinónimos comerciales o variaciones del mismo medicamento bajo distintos nombres.

In [14]:
# Normalizar composiciones (orden + limpieza)
df["Composition_clean"] = (
    df["Composition"]
    .str.lower()
    .str.split("+")
    .apply(lambda x: sorted([i.strip() for i in x]))
    .apply(lambda x: " + ".join(x))
)
# Agrupar por composición limpia
composition_counts = (
    df.groupby("Composition_clean")
    .size()
    .reset_index(name="count")
    .sort_values(by="count", ascending=False)
)

composition_counts.head(10)

,Composition_clean,count
2263,luliconazole (1% w/w),98
2163,levocetirizine (5mg) + montelukast (10mg),77
2053,ketoconazole (2% w/w),66
1360,domperidone (30mg) + rabeprazole (20mg),60
2032,itraconazole (100mg),53
2034,itraconazole (200mg),52
3148,telmisartan (40mg),52
1359,domperidone (30mg) + pantoprazole (40mg),47
307,amlodipine (5mg) + telmisartan (40mg),44
2347,metformin (500mg),44


In [15]:
# Ver nombres distintos con misma composición
# Crear dataframe en vez de serie
comp_summary = (
    df.groupby("Composition_clean")["Medicine Name"]
    .nunique()
    .reset_index(name="Cantidad de nombres distintos")
    .sort_values(by="Cantidad de nombres distintos", ascending=False)
)

comp_summary.head(10)

,Composition_clean,Cantidad de nombres distintos
2163,levocetirizine (5mg) + montelukast (10mg),76
2263,luliconazole (1% w/w),64
1360,domperidone (30mg) + rabeprazole (20mg),59
2053,ketoconazole (2% w/w),58
3148,telmisartan (40mg),51
2032,itraconazole (100mg),48
2034,itraconazole (200mg),47
1359,domperidone (30mg) + pantoprazole (40mg),47
307,amlodipine (5mg) + telmisartan (40mg),43
2347,metformin (500mg),42


In [16]:
# Obtener ejemplos de medicamentos por composición
comp_examples = (
    df.groupby("Composition_clean")["Medicine Name"]
    .unique()
    .reset_index()
)

# Convertir lista a string para mejor visualización
comp_examples["Medicamentos"] = comp_examples["Medicine Name"].apply(lambda x: ", ".join(x))

# Seleccionar columnas útiles
comp_examples = comp_examples[["Composition_clean", "Medicamentos"]]

comp_examples.head(5)

,Composition_clean,Medicamentos
0,abacavir (600mg) + lamivudine (300mg),Albavir Tablet
1,abiraterone acetate (250mg),"Abirapro 250mg Tablet, Xbira 250mg Tablet, Zel..."
2,abiraterone acetate (500mg),Zelgor 500mg Tablet
3,acamprosate (333mg),"Acamprol Tablet, Acamptas 333 Tablet"
4,acarbose (25mg),Glucobay 25 Tablet
